# Phase 1 Slice E: ViewState Export/Import

This notebook validates typed export/import flows:

- `POST /export/viewstate` returns full persisted `view_state`
- `POST /import/viewstate` creates a new persisted view with rebased identity
- CLI-style file handoff works by writing/reading inline JSON `view_state`


In [1]:
from __future__ import annotations

import base64
import json
import tempfile
from io import BytesIO
from pathlib import Path

import numpy as np
import zarr
from fastapi.testclient import TestClient
from PIL import Image

from lucida.client import LucidaClient
from lucida.server.app import create_app
from lucida.service.dataset_service import DatasetService


In [2]:
def create_sample_omezarr(uri: str) -> str:
    root = zarr.open_group(store=uri, mode='w')

    shape_level0 = (1, 2, 4, 8, 10)
    shape_level1 = (1, 2, 2, 4, 5)
    data_level0 = np.arange(np.prod(shape_level0), dtype=np.uint16).reshape(shape_level0)
    data_level1 = np.arange(np.prod(shape_level1), dtype=np.uint16).reshape(shape_level1)

    root.create_array('0', data=data_level0, chunks=(1, 1, 2, 4, 5), overwrite=True)
    root.create_array('1', data=data_level1, chunks=(1, 1, 1, 2, 3), overwrite=True)

    root.attrs['multiscales'] = [
        {
            'name': 'primary',
            'axes': [
                {'name': 't', 'type': 't'},
                {'name': 'c', 'type': 'c'},
                {'name': 'z', 'type': 'z'},
                {'name': 'y', 'type': 'y'},
                {'name': 'x', 'type': 'x'},
            ],
            'datasets': [
                {'path': '0', 'coordinateTransformations': [{'type': 'scale', 'scale': [1, 1, 1, 1, 1]}]},
                {'path': '1', 'coordinateTransformations': [{'type': 'scale', 'scale': [1, 1, 2, 2, 2]}]},
            ],
        }
    ]
    root.attrs['omero'] = {
        'channels': [
            {'index': 0, 'label': 'c0', 'color': 'ffffff', 'window': {'start': 0, 'end': 500}},
            {'index': 1, 'label': 'c1', 'color': 'ff0000', 'window': {'start': 0, 'end': 1500}},
        ]
    }
    return uri


In [3]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-export-import-'))
dataset_uri = create_sample_omezarr(str(tmp_dir / 'sample.zarr'))
view_state_file = tmp_dir / 'exported_view_state.json'

service = DatasetService()
app = create_app(dataset_service=service)
http_client = TestClient(app)
client = LucidaClient(client=http_client)

session = client.create_session()
opened = client.open_dataset(uri=dataset_uri, session_id=session.session_id)
created = client.create_view(dataset_id=opened.dataset_summary.dataset_id, session_id=session.session_id)

assert opened.schema_version == 1
assert created.view_state.mode == '2d'
assert created.view_state.view_id
created.view_state.view_id


'view_7ecf682178f445c2'

In [4]:
exported = client.export_viewstate(view_id=created.view_state.view_id, session_id=session.session_id)

assert exported.schema_version == 1
assert exported.export_id.startswith('exp_')
assert exported.source_view_id == created.view_state.view_id
assert exported.view_state.view_id == created.view_state.view_id
assert exported.view_state.state_hash
exported


ViewStateExportResponse(schema_version=1, export_id='exp_2c1750272c8b4f0b', exported_at=datetime.datetime(2026, 2, 24, 1, 26, 24, 433618, tzinfo=TzInfo(0)), source_view_id='view_7ecf682178f445c2', view_state=ViewState(schema_version=1, view_id='view_7ecf682178f445c2', session_id='session_0ee82ecfcf1642de', created_at=datetime.datetime(2026, 2, 24, 1, 26, 21, 83960, tzinfo=TzInfo(0)), mode='2d', datasets=[DatasetRef(dataset_id='ds_a48263e7493257d9', multiscale_name='primary')], viewport=Viewport(width_px=1024, height_px=1024, pixel_ratio=1.0), selectors=[AxisSelector(axis='t', kind='index', index=0, start=None, end_exclusive=None, indices=None, clamp=True), AxisSelector(axis='c', kind='index', index=0, start=None, end_exclusive=None, indices=None, clamp=True), AxisSelector(axis='z', kind='index', index=0, start=None, end_exclusive=None, indices=None, clamp=True)], view_2d=View2D(plane='xy', slice=SliceSettings(axis='z', index=0, slab=SlabSettings(thickness_vox=1, mode='single')), camera

In [5]:
raw_export = http_client.post(
    '/export/viewstate',
    json={
        'schema_version': 1,
        'view_id': created.view_state.view_id,
        'session_id': session.session_id,
    },
)
assert raw_export.status_code == 200
raw_export_payload = raw_export.json()
assert raw_export_payload['schema_version'] == 1
assert raw_export_payload['source_view_id'] == created.view_state.view_id
raw_export_payload['export_id']


'exp_f170857f5c8d46c2'

In [6]:
view_state_file.write_text(
    json.dumps(exported.view_state.model_dump(mode='json'), indent=2),
    encoding='utf-8',
)
assert view_state_file.exists()
stored_payload = json.loads(view_state_file.read_text(encoding='utf-8'))
assert stored_payload['view_id'] == created.view_state.view_id
str(view_state_file)


'/var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-export-import-h64vt1ek/exported_view_state.json'

In [7]:
imported = client.import_viewstate(
    view_state=stored_payload,
    session_id=session.session_id,
)

assert imported.schema_version == 1
assert imported.import_id.startswith('imp_')
assert imported.imported_from_view_id == created.view_state.view_id
assert imported.view_state.view_id != created.view_state.view_id
assert imported.view_state.state_version == 0
assert imported.view_state.state_hash
assert imported.selectors_applied
imported


ViewStateImportResponse(schema_version=1, import_id='imp_e34df99408e84643', imported_from_view_id='view_7ecf682178f445c2', view_state=ViewState(schema_version=1, view_id='view_09774b06bc6940a4', session_id='session_0ee82ecfcf1642de', created_at=datetime.datetime(2026, 2, 24, 1, 27, 46, 253952, tzinfo=TzInfo(0)), mode='2d', datasets=[DatasetRef(dataset_id='ds_a48263e7493257d9', multiscale_name='primary')], viewport=Viewport(width_px=1024, height_px=1024, pixel_ratio=1.0), selectors=[AxisSelector(axis='t', kind='index', index=0, start=None, end_exclusive=None, indices=None, clamp=True), AxisSelector(axis='c', kind='index', index=0, start=None, end_exclusive=None, indices=None, clamp=True), AxisSelector(axis='z', kind='index', index=0, start=None, end_exclusive=None, indices=None, clamp=True)], view_2d=View2D(plane='xy', slice=SliceSettings(axis='z', index=0, slab=SlabSettings(thickness_vox=1, mode='single')), camera=Camera2D(center_world=(5.0, 4.0), zoom=1.0, rotation_deg=0.0)), view_3d=

In [8]:
rendered = client.render_image(
    view_id=imported.view_state.view_id,
    session_id=session.session_id,
    width_px=96,
    height_px=64,
)

assert rendered.schema_version == 1
assert rendered.status == 'ok'
assert rendered.images[0].mime == 'image/png'
decoded = Image.open(BytesIO(base64.b64decode(rendered.images[0].bytes_base64))).convert('RGBA')
assert decoded.size == (96, 64)
rendered.meta.dataset_id


'ds_a48263e7493257d9'

In [9]:
client.close()
http_client.close()
'cleanup complete'


'cleanup complete'

## Expected output checks

Verify all checks below succeed:

- `schema_version == 1` for export/import/render responses.
- Export response includes full `view_state` with persisted identity/hash/version fields.
- Imported view has a new `view_id` and `state_version == 0`.
- Imported response includes non-empty `selectors_applied`.
- Rendering the imported view succeeds with PNG output at requested dimensions.
- File handoff (`view_state` JSON write/read) is valid for import flow.
